In [ ]:
# !pip install supervision ultralytics trackers
import supervision as sv
from ultralytics import YOLO, SAM
from trackers import ByteTrackTracker
import cv2
import numpy as np
import urllib.request
from pathlib import Path

Path("assets").mkdir(exist_ok=True)
urllib.request.urlretrieve(
    "https://media.roboflow.com/supervision/video-examples/vehicles.mp4",
    "assets/videos/vehicles.mp4"
)

yolo_model = YOLO("yolov8n.pt")
sam_model  = SAM("sam3.pt")
# sam3.pt ≈ 3.4 GB — si no se descarga automáticamente:
# https://huggingface.co/facebook/sam3
tracker    = ByteTrackTracker()

video_info = sv.VideoInfo.from_video_path("assets/videos/vehicles.mp4")
print(f"Resolución: {video_info.width} × {video_info.height}")
print(f"FPS: {video_info.fps} | Total frames: {video_info.total_frames}")

Resolución: 3840 × 2160
FPS: 25.0 | Total frames: 538


In [ ]:
mask_annotator  = sv.MaskAnnotator(opacity=0.6)
label_annotator = sv.LabelAnnotator()
trace_annotator = sv.TraceAnnotator()

tracker = ByteTrackTracker()

def procesar_frame_sam(frame: np.ndarray, _: int) -> np.ndarray:
    # Paso 1: Detectar con YOLO y asignar IDs persistentes
    yolo_results = yolo_model(frame, verbose=False)[0]
    yolo_det     = sv.Detections.from_ultralytics(yolo_results)
    yolo_det     = tracker.update(yolo_det)
    
    if len(yolo_det) == 0:
        return frame
    
    # Paso 2: SAM genera máscaras usando las cajas de YOLO como guía
    # .tolist() es necesario porque SAM espera lista de Python, no array NumPy
    bboxes      = yolo_det.xyxy.tolist()
    sam_results = sam_model(frame, bboxes=bboxes, verbose=False)[0]
    sam_det     = sv.Detections.from_ultralytics(sam_results)
    
    # Paso 3: Transferir atributos de YOLO a SAM
    # SAM preserva el orden de los bboxes de entrada — la alineación posicional es segura
    if len(sam_det) == len(yolo_det):
        sam_det.tracker_id = yolo_det.tracker_id
        sam_det.class_id   = yolo_det.class_id
        sam_det.confidence = yolo_det.confidence
    
    labels = [
        f"ID:{tid} {yolo_results.names[c]}"
        for tid, c in zip(sam_det.tracker_id, sam_det.class_id)
    ]
    
    annotated = mask_annotator.annotate(scene=frame.copy(), detections=sam_det)
    annotated = label_annotator.annotate(scene=annotated, detections=sam_det, labels=labels)
    annotated = trace_annotator.annotate(scene=annotated, detections=sam_det)
    return annotated

sv.process_video(
    source_path="assets/videos/vehicles.mp4",
    target_path="assets/videos/vehicles_sam.mp4",
    callback=procesar_frame_sam
)
print("Guardado: assets/videos/vehicles_sam.mp4")

In [ ]:
# Inspeccionamos un frame para ver qué contiene sam_det ANTES y DESPUÉS de la transferencia
tracker = ByteTrackTracker()
cap = cv2.VideoCapture("assets/videos/vehicles.mp4")
ret, frame_test = cap.read()
cap.release()

yolo_r   = yolo_model(frame_test, verbose=False)[0]
yolo_det = sv.Detections.from_ultralytics(yolo_r)
yolo_det = tracker.update(yolo_det)

bboxes   = yolo_det.xyxy.tolist()
sam_r    = sam_model(frame_test, bboxes=bboxes, verbose=False)[0]
sam_det  = sv.Detections.from_ultralytics(sam_r)

print("SAM ANTES de transferencia:")
print(f"  tracker_id: {sam_det.tracker_id}")   # None — SAM no sabe de tracking
print(f"  class_id:   {sam_det.class_id}")      # None o índice SAM interno
print(f"  mask shape: {sam_det.mask.shape if sam_det.mask is not None else None}")

if len(sam_det) == len(yolo_det):
    sam_det.tracker_id = yolo_det.tracker_id
    sam_det.class_id   = yolo_det.class_id
    sam_det.confidence = yolo_det.confidence

print("\nSAM DESPUÉS de transferencia:")
print(f"  tracker_id: {sam_det.tracker_id}")   # ahora tiene IDs de ByteTrack
print(f"  class_id:   {sam_det.class_id}")      # ahora tiene clases de YOLO

WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
SAM ANTES de transferencia:
  tracker_id: None
  class_id:   [0 1]
  mask shape: (2, 2160, 3840)

SAM DESPUÉS de transferencia:
  tracker_id: [-1 -1]
  class_id:   [2 2]


In [ ]:
# Combinamos PolygonZone (NB05) con SAM (NB06/NB07)
# Solo segmentamos los objetos que están dentro de la zona
POLYGON = np.array([
    [0,                         video_info.height // 2],
    [video_info.width // 2,     video_info.height // 2],
    [video_info.width // 2,     video_info.height],
    [0,                         video_info.height],
])
zone     = sv.PolygonZone(polygon=POLYGON)
zone_ann = sv.PolygonZoneAnnotator(zone=zone, color=sv.Color.RED, thickness=3)

tracker = ByteTrackTracker()

def callback_zona_sam(frame: np.ndarray, _: int) -> np.ndarray:
    yolo_r   = yolo_model(frame, verbose=False)[0]
    yolo_det = sv.Detections.from_ultralytics(yolo_r)
    yolo_det = tracker.update(yolo_det)
    
    # Filtrar solo objetos en la zona ANTES de llamar a SAM
    # — SAM es lento: no gastar tiempo en objetos fuera de interés
    en_zona  = zone.trigger(detections=yolo_det)
    yolo_det = yolo_det[en_zona]
    
    annotated = frame.copy()
    if len(yolo_det) > 0:
        bboxes   = yolo_det.xyxy.tolist()
        sam_r    = sam_model(frame, bboxes=bboxes, verbose=False)[0]
        sam_det  = sv.Detections.from_ultralytics(sam_r)
        if len(sam_det) == len(yolo_det):
            sam_det.tracker_id = yolo_det.tracker_id
            sam_det.class_id   = yolo_det.class_id
        annotated = mask_annotator.annotate(scene=annotated, detections=sam_det)
    
    annotated = zone_ann.annotate(scene=annotated)
    return annotated

sv.process_video(
    source_path="assets/videos/vehicles.mp4",
    target_path="assets/videos/vehicles_sam_zona.mp4",
    callback=callback_zona_sam
)
print("Guardado: assets/videos/vehicles_sam_zona.mp4")
# 💭 Reflexión: ¿Cuánto más rápido es este callback comparado con el que segmenta todo?
# Al filtrar antes de SAM, reducimos el número de objetos que SAM procesa por frame.

WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]


In [ ]:
from ultralytics.models.sam import SAM3VideoSemanticPredictor
import torch

overrides = dict(conf=0.25, task="segment", mode="predict", model="sam3.pt")
if torch.cuda.is_available():
    overrides["half"] = True

video_predictor = SAM3VideoSemanticPredictor(overrides=overrides)

# stream=True procesa frame a frame sin cargar todo el video en memoria
resultados_video = video_predictor(
    source="assets/videos/vehicles.mp4",
    text=["car", "bus", "truck"],
    stream=True
)

# Convertir y guardar frame a frame
import cv2
from pathlib import Path

cap    = cv2.VideoCapture("assets/vehicles.mp4")
fps    = cap.get(cv2.CAP_PROP_FPS)
w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

writer = cv2.VideoWriter(
    "assets/vehicles_texto.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
)

mask_ann = sv.MaskAnnotator(opacity=0.6)

for res in resultados_video:
    det      = sv.Detections.from_ultralytics(res)
    frame    = res.orig_img
    annotado = mask_ann.annotate(scene=frame.copy(), detections=det)
    writer.write(annotado)

writer.release()
print("Guardado: assets/videos/vehicles_texto.mp4")


In [ ]:
tracker = ByteTrackTracker()
predictor = SAM('sam3.pt')

def mi_callback(frame: np.ndarray, _: int) -> np.ndarray:
    yolo_r   = yolo_model(frame, verbose=False)[0]
    yolo_det = sv.Detections.from_ultralytics(yolo_r)
    yolo_det = tracker.update(yolo_det)
    mask_valida = yolo_det.tracker_id <= 5        # máscara booleana: primeros 5 IDs
    primeros     = yolo_det[mask_valida]           # sv.Detections filtrado
    if len(primeros) == 0:
        return frame.copy()
    sam_r = predictor(bboxes=primeros.xyxy)[0]
    sam_det = sv.Detections.from_ultralytics(sam_r)
    annotated = mask_annotator.annotate(scene=frame.copy(), detections=sam_det)
    return annotated